In [2]:
import numpy as np
import pandas as pd
import os
from urllib.parse import unquote
import openai
import random
# from openai import version
from time import sleep
import pickle
import json
import copy
import re
import regex
PATH_IN = '../../data/wikispeedia/wikispeedia_paths-and-graph/'

%load_ext autoreload
%autoreload 2

random.seed(42)

In [3]:
fnodes = open(os.path.join(PATH_IN, 'articles.tsv'))
nodes = {}
for i, line in enumerate(fnodes):
    line = line.strip()
    if '#' in line or line == '':
        continue
    node = unquote(line)
    nodes[node]=True

In [4]:
len(nodes)

4604

In [5]:
nodes['Jurassic']

True

In [6]:
flinks = open(os.path.join(PATH_IN, 'links.tsv'))
links = {}
for i, line in enumerate(flinks):
    line = line.strip()
    if '#' in line or line == '':
        continue
    line = line.split('\t')
    src = unquote(line[0]); tgt = unquote(line[1])
    if src not in links:
        links[src] = [tgt]
    else:
        links[src].append(tgt)

In [7]:
";".join(links['Éire'])

'Canada;English_language;George_VI_of_the_United_Kingdom;Ireland;Northern_Ireland;Ottawa;Republic_of_Ireland;United_Kingdom'

In [8]:
pairs_finished = {}; pairs_unfinished = {}; paths_real = []

In [9]:
fpaths_finished = open(os.path.join(PATH_IN, 'paths_finished.tsv'))
for i, line in enumerate(fpaths_finished):
    line = line.strip()
    if '#' in line or line == '':
        continue
    line = line.split('\t')
    path = line[3].split(';')
    src = unquote(path[0]); tgt = unquote(path[-1])
    path_unquote = []
    for article in path:
        path_unquote.append(unquote(article))
    if (src,tgt) not in pairs_finished:
        pairs_finished[(src,tgt)] = True
    paths_real.append([src,tgt,";".join(path_unquote),len(path_unquote)-1,1])

In [10]:
fpaths_unfinished = open(os.path.join(PATH_IN, 'paths_unfinished.tsv'))
for i, line in enumerate(fpaths_unfinished):
    line = line.strip()
    if '#' in line or line == '':
        continue
    line = line.split('\t')
    path = line[3].split(';')
    src = unquote(path[0]); tgt = unquote(line[4])
    path_unquote = []
    for article in path:
        path_unquote.append(unquote(article))
    if (src,tgt) not in pairs_unfinished and (src,tgt) not in pairs_finished:
        pairs_unfinished[(src,tgt)] = True
    paths_real.append([src,tgt,";".join(path_unquote),len(path_unquote)-1,0])

In [11]:
paths_real_df = pd.DataFrame(paths_real, columns=['src', 'tgt', 'path', 'path_len', 'is_path_finished'])
paths_real_df

,src,tgt,path,path_len,is_path_finished
0,14th_century,African_slave_trade,14th_century;15th_century;16th_century;Pacific...,8,1
1,14th_century,African_slave_trade,14th_century;Europe;Africa;Atlantic_slave_trad...,4,1
2,14th_century,African_slave_trade,14th_century;Niger;Nigeria;British_Empire;Slav...,7,1
3,14th_century,Greece,14th_century;Renaissance;Ancient_Greece;Greece,3,1
4,14th_century,John_F._Kennedy,14th_century;Italy;Roman_Catholic_Church;HIV;R...,6,1
...,...,...,...,...,...
76188,Franz_Kafka,Cholera,Franz_Kafka;Tuberculosis;World_Health_Organiza...,7,0
76189,Modern_history,Hollandic,Modern_history,0,0
76190,Computer_programming,The_Beatles,Computer_programming;Linguistics;Culture;Popul...,4,0
76191,Jamaica,Alan_Turing,Jamaica;United_Kingdom;World_War_II;Battle_of_...,3,0


In [12]:
len(pairs_finished), len(pairs_unfinished)

(28718, 13912)

In [13]:
paths_real_finished_sample = paths_real_df.query('is_path_finished == 1 and path_len <= 10').sample(n=100, random_state=42)
paths_real_unfinished_sample = paths_real_df.query('is_path_finished == 0').sample(n=10, random_state=42)

In [14]:
paths_real_finished_sample

,src,tgt,path,path_len,is_path_finished
5715,Jurassic,Cologne,Jurassic;Germany;Cologne,2,1
14457,Brain,Telephone,Brain;Computer_science;Society;<;Internet;Info...,7,1
29624,Honey,"September_11,_2001_attacks","Honey;United_States;September_11,_2001_attacks",2,1
39628,Great_Yarmouth,Salsa_music,Great_Yarmouth;World_War_II;United_States;Mexi...,5,1
32444,Anna_Karenina,Age_of_Enlightenment,Anna_Karenina;Saint_Petersburg;Jew;Age_of_Enli...,3,1
...,...,...,...,...,...
24696,Apricot,Nintendo_Entertainment_System,Apricot;China;Japan;Electronics;Computer;Compu...,6,1
26513,Venezuela,Thermodynamics,Venezuela;Petroleum;Global_warming;Nuclear_pow...,7,1
35549,Belize,Zebra_shark,Belize;Caribbean_Sea;Fishing;Shark;Zebra_shark,4,1
45992,Richard_Nixon,Tropical_cyclone,Richard_Nixon;California;Pacific_Ocean;Atlanti...,4,1


In [15]:
paths_real_unfinished_sample

,src,tgt,path,path_len,is_path_finished
65822,Orion_Nebula,Pi,Orion_Nebula;Telescope;<;Universe;Euclidean_ge...,8,0
73415,Ant,Hurricane_Katrina,Ant;Earth;United_Nations;United_States,3,0
63206,Magma,16_Cygni,Magma,0,0
74884,King_Arthur,Cosmic_microwave_background_radiation,King_Arthur;Great_Britain;England;Charles_Darw...,6,0
69448,Barack_Obama,Cheese,Barack_Obama;Chicago;Season;Temperate;Geograph...,8,0
65916,Joan_of_Arc,Italo_disco,Joan_of_Arc;Giuseppe_Verdi;Italy,2,0
52921,Thomas_Jefferson,Gazelle,Thomas_Jefferson;France;Sub-Saharan_Africa;Mad...,4,0
66174,Cookiecutter_shark,Literacy,Cookiecutter_shark;Scientific_classification;L...,5,0
54991,Theatre,Zebra,Theatre;Actor;Film,2,0
66682,Brain,Telephone,Brain;Animal,1,0


In [16]:
path_demonstrations = []
paths_real_sample = paths_real_df.query('is_path_finished == 1').sample(n=20, random_state=101)[['src', 'tgt', 'path']].values.tolist()
for path in paths_real_sample:
    path_obj = {}
    path_obj['src'] = path[0]; path_obj['tgt'] = path[1]
    path_obj['output'] = path[2]
    path_demonstrations.append(path_obj)
path_demonstrations

[{'src': 'Ottoman_Empire',
  'tgt': 'Rosetta_Stone',
  'output': 'Ottoman_Empire;Egypt;Ancient_Egypt;Rosetta_Stone'},
 {'src': 'Thomas_Cranmer',
  'tgt': 'Computer_and_video_games',
  'output': 'Thomas_Cranmer;William_Shakespeare;England;Time_zone;Telecommunication;Internet;Microsoft;Television;Computer_and_video_games'},
 {'src': 'American_Revolutionary_War',
  'tgt': 'Bison',
  'output': 'American_Revolutionary_War;United_States;Yellowstone_National_Park;Cattle;Bison'},
 {'src': 'Omega-3_fatty_acid',
  'tgt': 'Chew_Valley',
  'output': 'Omega-3_fatty_acid;Schizophrenia;United_Kingdom;England;Bristol;<;History_of_Anglo-Saxon_England;<;<;England;River_Severn;<;<;England;John_Locke;Chew_Magna;Chew_Valley'},
 {'src': 'The_Supremes',
  'tgt': 'Iron_Age',
  'output': 'The_Supremes;Chicago;Industry;Industrial_Revolution;<;Steel;Iron;<;Bronze_Age;Iron_Age'},
 {'src': 'Owl', 'tgt': 'Penguin', 'output': 'Owl;Antarctica;Penguin'},
 {'src': 'Weed',
  'tgt': 'Clifton_Suspension_Bridge',
  'output

In [17]:
click_demonstrations = []
paths_real_sample = paths_real_df.query('is_path_finished == 1').sample(n=20, random_state=101)[['src', 'tgt', 'path']].values.tolist()
for path in paths_real_sample:
    path_obj = {}
    path_obj['src'] = path[0]; path_obj['tgt'] = path[1]
    path_obj['cands'] = ";".join(links[path_obj['src']])
    path_obj['output'] = path[2].split(";")[1]
    click_demonstrations.append(path_obj)
click_demonstrations

[{'src': 'Ottoman_Empire',
  'tgt': 'Rosetta_Stone',
  'cands': "15th_century;16th_century;17th_century;20th_century;Abbasid;Albania;Algeria;Arabic_language;Armenia;Asia;Atlantic_Ocean;Austria;Azerbaijan;Baghdad;Baroque;Beirut;Black_Sea;Bulgaria;Byzantine_Empire;Calligraphy;Caspian_Sea;Christopher_Columbus;Constitutional_monarchy;Crimean_War;Currency;Cyprus;Egypt;England;Europe;Folk_music;France;Greece;Greek_War_of_Independence;Guild;Habsburg_Spain;Holy_Roman_Empire;Indian_Ocean;Islam;Istanbul;Jerusalem;Johannes_Gutenberg;Libya;List_of_countries_by_system_of_government;Marco_Polo;Mediterranean_Sea;Mesopotamia;Middle_Ages;Middle_East;Moldova;Monarchy;Mongol_Empire;Montenegro;Muhammad;Musical_instrument;Nationalism;North_Africa;Persian_Gulf;Piano;Poland;Portugal;Qur'an;Red_Sea;Rococo;Roman_Empire;Romania;Russia;Russian_Revolution_of_1917;Sarajevo;Serbia;Siege;Slavery;Slovakia;Spanish_Inquisition;Sudan;Suleiman_the_Magnificent;Sultan;The_Holocaust;Tunis;Turkey;Ukraine;University;Vienna;Wo

In [18]:
click_demonstrations_with_memory = []
paths_real_sample = paths_real_df.query('is_path_finished == 1').sample(n=20, random_state=101)[['src', 'tgt', 'path']].values.tolist()
random.seed(42)
for i, path in enumerate(paths_real_sample):
    path_obj = {}
    path_obj['src'] = path[0]; path_obj['tgt'] = path[1]
    path_list = path[2].split(";")
    cur_idx = random.randint(0,len(path_list)-2)
    if i==19:
        cur_idx=0
    # print(cur_idx, len(path_list))
    path_obj['cur'] = path_list[cur_idx]
    if path_obj['cur'] == "<":
        # print("Cur=<", path)
        path_obj['cur'] = 'United_Kingdom'
        path_obj['cands'] = ";".join(links['United_Kingdom'])
        path_obj['path_so_far'] = ";".join(['Omega-3_fatty_acid', 'Schizophrenia', 'United_Kingdom', 'England', 'Bristol', 'England', 'History_of_Anglo-Saxon_England', 'England', 'United_Kingdom'])
    else:
        path_obj['cands'] = ";".join(links[path_obj['cur']])
        if i==17:
            path_obj['path_so_far'] = ";".join(['Samuel_Beckett', 'Miami,_Florida', 'Hurricane_Andrew', 'Tropical_cyclone', 'Hurricane_Andrew', 'Miami,_Florida', 'Hurricane_Andrew', 'Miami,_Florida', 'Hurricane_Andrew'])    
        else:
            path_obj['path_so_far'] = ";".join(path_list[:cur_idx+1])
    if path_list[cur_idx+1] == "<":
        # print("Cur+1=<", path)
        if path_obj['cur']=='Industrial_Revolution':
            path_obj['output'] = "Iron"
        elif path_obj['cur']=='Jew':
            path_obj['output'] = "Judaism"
    else:
        path_obj['output'] = path_list[cur_idx+1]
    click_demonstrations_with_memory.append(path_obj)
click_demonstrations_with_memory

[{'src': 'Ottoman_Empire',
  'tgt': 'Rosetta_Stone',
  'cur': 'Ancient_Egypt',
  'cands': '15th_century;4th_century;Africa;Agriculture;Alexander_the_Great;Ancient_Greece;Ancient_Rome;Anemia;Anno_Domini;Archaeology;Beekeeping;Beer;Berber_languages;Board_game;Boat;Carbon;Cereal;Chess;Christianity;Civilization;Copper;Culture;Currency;East_Africa;Egypt;Egyptian_pyramids;Electricity;Empiricism;Engineering;Fishing;Geology;Geometry;Glass;Granite;Great_Pyramid_of_Giza;Herodotus;Homer;Horn_of_Africa;Iron;Iron_Age;Irrigation;Islam;Literature;Mathematics;Mediterranean_Sea;Middle_Ages;Mining;Mummy;Nile;North_Africa;Odysseus;Property;Pyramid;Red_Sea;Religion;Roman_Empire;Rosetta_Stone;Sahara;Society;Steel;Sudan;Suez_Canal;Turquoise;Virgil;Vowel;Wine;Writing',
  'path_so_far': 'Ottoman_Empire;Egypt;Ancient_Egypt',
  'output': 'Rosetta_Stone'},
 {'src': 'Thomas_Cranmer',
  'tgt': 'Computer_and_video_games',
  'cur': 'William_Shakespeare',
  'cands': 'Actor;Ben_Jonson;British_Empire;Employment;England

In [23]:
payload = {
#     "conf1": {
#       "max_tokens": 20,
#       "temperature": 1,
#       "top_p": 1,
#     },
#     "conf2": {
#       "max_tokens": 20,
#       "temperature": 0.7,
#       "top_p": 1,
#     },
    "conf1": {
      "max_tokens": 256,
      "temperature": 1,
      "top_p": 1,
      "request_timeout": 30,
    },
    "conf2": {
      "max_tokens": 256,
      "temperature": 0.75,
      "top_p": 1,
      "request_timeout": 30,
    },
    "conf3": {
      "max_tokens": 256,
      "temperature": 0.5,
      "top_p": 1,
      "request_timeout": 30,
    },
    "conf4": {
      "max_tokens": 256,
      "temperature": 0.25,
      "top_p": 1,
      "request_timeout": 30,
    },
    "conf5": {
      "max_tokens": 256,
      "temperature": 0.1,
      "top_p": 1,
      "request_timeout": 30,
    }
}

In [27]:
def navigate_entire_path(src, tgt, path_demonstrations, conf='conf1', fewshot=5, n=1):
    # Current system prompt
    demonstrations = [{"role":"system", "content": 
                 f'''You are a knowledge discovery expert familiar with the Wikipedia link structure and your objective is to play the game of Wikispeedia: https://dlab.epfl.ch/wikispeedia/play/.'''}]

    # Current initial prompt for describing the task
    demonstrations.append({"role": "user", "content": '''##Goal \nGiven two Wikipedia articles, a source and a target, your goal is to reach the target article starting from the source article in as few clicks as possible. For the articles you are given this is always possible.\n
    ##Constraint \nYou should exclusively follow the links present in the articles that you encounter along the way.\n
    ##Strategy Use your knowledge of the "expected" Wikipedia link structure and relatedness between articles to identify a path from the source article to the target article.\n
    ##Input \nYou will be given input in the JSON format as described below\n
    Input format:
    {
        "Source": "<source>", 
        "Target": "<target>"
    }

    ##Output \nYou should only respond in the JSON format as described below\n
    Output format:
    {
        "Output": "<output path with articles separated by ';'>"
    }
    Ensure the response can be parsed by Python json.loads
    '''})

    # Include 5 few-shot demonstrations for ensuring that the model understands the input and output structure
    if fewshot:
        # always the same demonstrations
        for demo in path_demonstrations[0:fewshot]:
            demonstrations.append({"role": "user", "content": f'''{{"Source": "{demo['src']}", "Target": "{demo['tgt']}"}}'''})
            demonstrations.append({"role": "assistant", "content": f'''{{"Output": "{demo['output']}"}}'''})
    
    messages = []
    messages.extend(demonstrations)
    messages.append({"role": "user", "content": f'''{{"Source": "{src}", "Target": "{tgt}"}}'''})
    return messages
#     print(messages[-1])
#     print()

    # setting n very high is almost free.
    # the model only generates 2-3 tokens for each reply and we only pay for the demonstrations once
    # n=128 is the highest the API supports
    # response = openai.ChatCompletion.create(model="gpt-3.5-turbo", **payload[conf], messages=messages, n=n)

    # # parsing the guesses
    # response_text = [c['message']['content'] for c in response['choices']]
    # return response_text, response.usage.total_tokens

In [28]:
messages = navigate_entire_path('United_States', 'Gold(III)_chloride', path_demonstrations, conf='conf1', fewshot=5, n=1)
for msg in messages:
    print(msg['content'])

You are a knowledge discovery expert familiar with the Wikipedia link structure and your objective is to play the game of Wikispeedia: https://dlab.epfl.ch/wikispeedia/play/.
##Goal 
Given two Wikipedia articles, a source and a target, your goal is to reach the target article starting from the source article in as few clicks as possible. For the articles you are given this is always possible.

    ##Constraint 
You should exclusively follow the links present in the articles that you encounter along the way.

    ##Strategy Use your knowledge of the "expected" Wikipedia link structure and relatedness between articles to identify a path from the source article to the target article.

    ##Input 
You will be given input in the JSON format as described below

    Input format:
    {
        "Source": "<source>", 
        "Target": "<target>"
    }

    ##Output 
You should only respond in the JSON format as described below

    Output format:
    {
        "Output": "<output path with art

In [25]:
def navigate_entire_path_with_plan(src, tgt, path_demonstrations, conf='conf1', fewshot=5, n=1):
    # Current system prompt
    demonstrations = [{"role":"system", "content": 
                 f'''You are a knowledge discovery expert familiar with the Wikipedia link structure and your objective is to play the game of Wikispeedia: https://dlab.epfl.ch/wikispeedia/play/.'''}]

    # Current initial prompt for describing the task
    demonstrations.append({"role": "user", "content": '''##Goal \nGiven two Wikipedia articles, a source and a target, your goal is to reach the target article starting from the source article in as few clicks as possible. For the articles you are given this is always possible.\n
    ##Constraint \nYou should exclusively follow the links present in the articles that you encounter along the way.\n
    ##Strategy Use your knowledge of the "expected" Wikipedia link structure and relatedness between articles to identify a path from the source article to the target article.\n
    ##Input \nYou will be given input in the JSON format as described below\n
    Input format:
    {
        "Source": "<source>", 
        "Target": "<target>"
    }

    ##Output \nYou should only respond in the JSON format as described below\n
    Output format:
    {
        "Thoughts": {
            "Strategy": "<general strategy employed to reach the target>",
            "Detailed reasoning": "<- short bulleted- list explaining the- output path step-by-step>"
        }, 
        "Output": "<output path with articles separated by ';'>"
    }
    Ensure the response can be parsed by Python json.loads
    '''})

    # Include 5 few-shot demonstrations for ensuring that the model understands the input and output structure
    if fewshot:
        # always the same demonstrations
        for demo in path_demonstrations[0:fewshot]:
            demonstrations.append({"role": "user", "content": f'''{{"Source": "{demo['src']}", "Target": "{demo['tgt']}"}}'''})
            demonstrations.append({"role": "assistant", "content": f'''{{"Output": "{demo['output']}"}}'''})

    messages = []
    messages.extend(demonstrations)
    messages.append({"role": "user", "content": f'''{{"Source": "{src}", "Target": "{tgt}"}}'''})
#     print(messages[-1])
#     print()

    # setting n very high is almost free.
    # the model only generates 2-3 tokens for each reply and we only pay for the demonstrations once
    # n=128 is the highest the API supports
    response = openai.ChatCompletion.create(model="gpt-3.5-turbo", **payload[conf], messages=messages, n=n)

    # # parsing the guesses
    response_text = [c['message']['content'] for c in response['choices']]
    return response_text, response.usage.total_tokens

In [26]:
def wikify_article_name(name):
    return "_".join(name.strip().split(" "))

In [27]:
def check_path(src, tgt, generated_path, links):
    gen_path_list = []
    for node in generated_path.split(";"):
        gen_path_list.append(wikify_article_name(node))
    if gen_path_list[0] != src:
        return gen_path_list, 0, f"Generated source {gen_path_list[0]} not in the list of Wikispeedia articles"
    for i in range(len(gen_path_list)-1):
        cur = gen_path_list[i]; nxt = gen_path_list[i+1]
        if nxt not in links[cur]:
            return gen_path_list, 0, f"Generated click from {cur} to {nxt} is incorrect. Correct path prefix {';'.join(gen_path_list[:i+1])}"
    if gen_path_list[-1] == tgt:
        return gen_path_list, 1, "Sucess"
    else:
        return gen_path_list, 0, "Valid path, but couldn't reach the target"

In [28]:
def navigate_path_wrapper(src, tgt, path_demonstrations, conf, reasoning=False, fewshot=5):
    total_tokens_used =0; num_tries = 0
    while(num_tries<3):
        try:
            if reasoning:
                response, tokens_used = navigate_entire_path_with_plan(src, tgt, path_demonstrations, conf, fewshot)
            else:
                response, tokens_used = navigate_entire_path(src, tgt, path_demonstrations, conf, fewshot)
            total_tokens_used += tokens_used
            num_tries+=1
            #print("RAW Response", response)
            response_json_obj = extract_json_from_response(response[0])
            #print("JSON extracted from response", response_json_obj)
            if response_json_obj:
                gen_path_obj = json.loads(response_json_obj[0])
                if 'Output' in gen_path_obj:
                    gen_path = gen_path_obj['Output']
                    if reasoning:
                        gen_reason = gen_path_obj['Thoughts']
                else:
                    gen_path="NA"
                    if reasoning:
                        gen_reason="NA"
                    print(gen_path_obj)
            else:
                gen_path="NA"
                if reasoning:
                    gen_reason="NA"
                print(response_json_obj)
            if gen_path!='NA':
                break
        except (openai.error.RateLimitError, openai.error.APIError, openai.error.Timeout, openai.error.ServiceUnavailableError) as e1:
            print(f'Encountered an exception {e1}, sleeping')
            print(num_tries)
            sleep(10)
            gen_path='NA'
            if reasoning:
                gen_reason="NA"
        except ValueError as e2:
            print(f'Encountered an exception {e2}')
            print(response[0])
            print(num_tries)
            gen_path='NA'
            if reasoning:
                gen_reason="NA"
    if reasoning:
        return gen_path, gen_reason, total_tokens_used
    else:
        return gen_path, total_tokens_used

In [232]:
reasoning=False
for fewshot in [0, 10]:
    print(f'Fewshot demonstrations={fewshot}')
    paths_generated = []
    for index, row in paths_real_finished_sample.iterrows():
        print(row['src'], row['tgt'], row['path_len'])
        for conf in ['conf1', 'conf2', 'conf3', 'conf4', 'conf5']:
            attempt=0
            attempt+=1
            links_dict = copy.deepcopy(links)
            gen_path, num_tokens = navigate_path_wrapper(row['src'], row['tgt'], path_demonstrations, conf, reasoning, fewshot)
            gen_path_list, is_finished, detailed_outcome = check_path(row['src'], row['tgt'], gen_path, links_dict)
            if is_finished:
                print(gen_path_list, 'Successful', detailed_outcome, num_tokens, attempt, payload[conf]['temperature'], fewshot)
                paths_generated.append([row['src'], row['tgt'], row['path'], row['path_len'], gen_path_list, len(gen_path_list)-1, is_finished, detailed_outcome, num_tokens, num_tokens*0.002/1000, attempt, payload[conf]['temperature'], fewshot])
            else:
                print(gen_path_list, row['tgt'], 'Unsuccessful', detailed_outcome, num_tokens, attempt, payload[conf]['temperature'], fewshot)
                paths_generated.append([row['src'], row['tgt'], row['path'], row['path_len'], gen_path_list, len(gen_path_list)-1, is_finished, detailed_outcome, num_tokens, num_tokens*0.002/1000, attempt, payload[conf]['temperature'], fewshot])
    paths_generated_df = pd.DataFrame(paths_generated, columns=['src', 'tgt', 'human_path', 'human_path_len', 'generated_path', 'generated_path_len', 'is_finished', 'detailed_outcome', 'num_tokens', 'USD', 'attempt', 'temperature', 'num_demonstrations'])
    paths_generated_df.to_csv(f'generated_paths_entire_reasoning_{reasoning}_demonstrations_{fewshot}.csv', index=False)

Fewshot demonstrations=0
Jurassic Cologne 2
['Jurassic', 'Cologne'] Cologne Unsuccessful Generated click from Jurassic to Cologne is incorrect. Correct path prefix Jurassic 268 1 1 0
['Jurassic', 'Cologne'] Cologne Unsuccessful Generated click from Jurassic to Cologne is incorrect. Correct path prefix Jurassic 268 1 0.75 0
['Jurassic', 'Cologne'] Cologne Unsuccessful Generated click from Jurassic to Cologne is incorrect. Correct path prefix Jurassic 268 1 0.5 0
['Jurassic', 'Jurassic_Park', 'Cologne'] Cologne Unsuccessful Generated click from Jurassic to Jurassic_Park is incorrect. Correct path prefix Jurassic 272 1 0.25 0
['Jurassic', 'Cologne'] Cologne Unsuccessful Generated click from Jurassic to Cologne is incorrect. Correct path prefix Jurassic 268 1 0.1 0
Brain Telephone 7
['Brain', 'Computer', 'Telephone'] Telephone Unsuccessful Generated click from Brain to Computer is incorrect. Correct path prefix Brain 266 1 1 0
Encountered an exception Request timed out: HTTPSConnectionPool

In [35]:
reasoning=True
# for fewshot in [0, 10]:
for fewshot in [0]:
    print(f'Fewshot demonstrations={fewshot}')
    paths_generated = []
    for index, row in paths_real_finished_sample.iterrows():
        print(row['src'], row['tgt'], row['path_len'])
        for conf in ['conf1', 'conf2', 'conf3', 'conf4', 'conf5']:
            attempt=0
            attempt+=1
            links_dict = copy.deepcopy(links)
            gen_path, gen_reason, num_tokens = navigate_path_wrapper(row['src'], row['tgt'], path_demonstrations, conf, reasoning, fewshot)
            gen_path_list, is_finished, detailed_outcome = check_path(row['src'], row['tgt'], gen_path, links_dict)
            if is_finished:
                print(gen_path_list, gen_reason, 'Successful', detailed_outcome, num_tokens, attempt, payload[conf]['temperature'], fewshot)
                paths_generated.append([row['src'], row['tgt'], row['path'], row['path_len'], gen_path_list, len(gen_path_list)-1, gen_reason, is_finished, detailed_outcome, num_tokens, num_tokens*0.002/1000, attempt, payload[conf]['temperature'], fewshot])
            else:
                print(gen_path_list, gen_reason, row['tgt'], 'Unsuccessful', detailed_outcome, num_tokens, attempt, payload[conf]['temperature'], fewshot)
                paths_generated.append([row['src'], row['tgt'], row['path'], row['path_len'], gen_path_list, len(gen_path_list)-1, gen_reason, is_finished, detailed_outcome, num_tokens, num_tokens*0.002/1000, attempt, payload[conf]['temperature'], fewshot])
    paths_generated_df = pd.DataFrame(paths_generated, columns=['src', 'tgt', 'human_path', 'human_path_len', 'generated_path', 'generated_path_len', 'generated_path_thoughts', 'is_finished', 'detailed_outcome', 'num_tokens', 'USD', 'attempt', 'temperature', 'num_demonstrations'])
    paths_generated_df.to_csv(f'31Aug_generated_paths_entire_reasoning_{reasoning}_demonstrations_{fewshot}.csv', index=False)

Fewshot demonstrations=0
Jurassic Cologne 2
['Jurassic', 'Dinosaur', 'Cologne'] {'Strategy': 'I will use a breadth-first search (BFS) algorithm to find the shortest path between the source and target articles.', 'Detailed reasoning': ["Start with the source article, 'Jurassic'.", "Find all the articles that can be reached from 'Jurassic' and add them to a queue.", "Check if any of the articles in the queue match the target article, 'Cologne'. If so, we have found the shortest path.", 'If not, continue expanding the search by adding the linked articles of the articles in the queue to the queue.', 'Repeat this process until the target article is found.']} Cologne Unsuccessful Generated click from Dinosaur to Cologne is incorrect. Correct path prefix Jurassic;Dinosaur 458 1 1 0
['Jurassic', 'Cretaceous', 'Cologne'] {'Strategy': 'I will try to find a path from the source article to the target article by following the Wikipedia link structure. I will start by exploring the links in the sour

KeyboardInterrupt: 

In [ ]:
i = 0
for index, row in paths_real_finished_sample.iterrows():
    if i==5:
        break
    print(row['src'], row['tgt'], row['path_len'])
    for conf in ['conf_test1', 'conf_test2']:
        response, tokens_used = navigate_plan(row['src'], row['tgt'], conf)
        print(payload[conf])
        print("RAW Response", response)
        response_json_obj = extract_json_from_response(response[0])
        print("JSON extracted from response", response_json_obj)
        try:
            gen_click_obj = json.loads(response[0])['Output']
        except ValueError as e:
            print(e, gen_click_obj)
        print(tokens_used)
        print()
    i+=1

In [30]:
def navigate_naive(src, tgt, click_demonstrations, cands, conf='conf1', fewshot=5, backtrack=True, n=1):
    # Old system prompt
    # demonstrations = [{"role":"system", "content": 
    #              f'''You are an assistant that helps readers to navigate from one Wikipedia article to another. Following are some instructions: 
    #              1. You would be provided two Wikipedia articles, a source and a target. 
    #              2. Starting from the source article, your goal is to reach the target article, exclusively by following the links present in the articles that you encounter along the way.
    #              3. For each article that corresponds to your current state, links that can be clicked would be provided as candidates.
    #              4. You are encouraged to find shorter paths, but not at the expense of quality.
    #              '''}]

    # Current system prompt
    demonstrations = [{"role":"system", "content": 
                 f'''You are a knowledge discovery expert familiar with the Wikipedia link structure and your objective is to play the game of Wikispeedia: https://dlab.epfl.ch/wikispeedia/play/.'''}]

    # Old initial prompt for describing the task
    # demonstrations.append({"role": "user", "content": '''Given a source article, outgoing links from the source article as candidates, and a target article, your task is to select a link from the provided candidates that takes you closer to the target. 
    # If none of the provided candidates takes you closer to the target article, select "NA".'''})

    # Current initial prompt for describing the task
    if not backtrack: #without backtracking
        demonstrations.append({"role": "user", "content": '''##Goal \nGiven two Wikipedia articles, a source and a target, your goal is to reach the target article starting from the source article in as few clicks as possible. For the articles you are given this is always possible.\n
        ##Constraint \nYou should exclusively follow the links present in the articles that you encounter along the way.\n
        ##Fine-grained instructions \n1. While the overall goal is to find a path from a source to a target article, you will proceed step by step.\n2. Given outgoing links from the source article as candidates, you should select the candidate that takes you closer to the target article. Use your knowledge of the "expected" Wikipedia link structure and relatedness between articles to identify the candidate that takes you closer to the target.\n3. Choose **only** from the provided candidates.\n4. Do not provide an algorithm or code to solve the task, instead present only the solution.\n
        ##Input \nYou will be given input in the JSON format as described below\n
        Input format:
        {
            "Source": "<source>", 
            "Candidates": "<candidates>", 
            "Target": "<target>"
        }

        ##Output \nYou should only respond in the JSON format as described below\n
        Output format:
        {
            "Output": "<output>"
        }
        Ensure the response can be parsed by Python json.loads
        '''})
    else: #with backtracking
        demonstrations.append({"role": "user", "content": '''##Goal \nGiven two Wikipedia articles, a source and a target, your goal is to reach the target article starting from the source article in as few clicks as possible. For the articles you are given this is always possible.\n
        ##Constraint \nYou should exclusively follow the links present in the articles that you encounter along the way.\n
        ##Fine-grained instructions \n1. While the overall goal is to find a path from a source to a target article, you will proceed step by step.\n2. Given outgoing links from the source article as candidates, you should select the candidate that takes you closer to the target article. Use your knowledge of the "expected" Wikipedia link structure and relatedness between articles to identify the candidate that takes you closer to the target.\n3. If none of the provided candidates takes you closer to the target article, select "NA".\n4. Choose **only** from the provided candidates.\n5. Do not provide an algorithm or code to solve the task, instead present only the solution.\n
        ##Input \nYou will be given input in the JSON format as described below\n
        Input format:
        {
            "Source": "<source>", 
            "Candidates": "<candidates>", 
            "Target": "<target>"
        }

        ##Output \nYou should only respond in the JSON format as described below\n
        Output format:
        {
            "Output": "<output>"
        }
        Ensure the response can be parsed by Python json.loads
        '''})

    # Include 5 few-shot demonstrations for ensuring that the model understands the input and output structure
    if fewshot:
        # always the same demonstrations
        for demo in click_demonstrations[0:fewshot]:
            demonstrations.append({"role": "user", "content": f'''{{"Source": "{demo['src']}", "Candidates": "{demo['cands']}", "Target": "{demo['tgt']}"}}'''})
            demonstrations.append({"role": "assistant", "content": f'''{{"Output": "{demo['output']}"}}'''})

    messages = []
    messages.extend(demonstrations)
    messages.append({"role": "user", "content": f'''{{"Source": "{src}", "Candidates": "{";".join(cands)}", "Target": "{tgt}"}}'''})
    return messages
# #     print(messages[-1])
# #     print()

#     # setting n very high is almost free.
#     # the model only generates 2-3 tokens for each reply and we only pay for the demonstrations once
#     # n=128 is the highest the API supports
#     if fewshot>=20:
#         response = openai.ChatCompletion.create(model="gpt-3.5-turbo-16k", **payload[conf], messages=messages, n=n)
#     else:
#         response = openai.ChatCompletion.create(model="gpt-3.5-turbo", **payload[conf], messages=messages, n=n)

#     # # parsing the guesses
#     response_text = [c['message']['content'] for c in response['choices']]
#     return response_text, response.usage.total_tokens

In [40]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("/dlabdata1/llama2_hf/Llama-2-7b-chat-hf/")
messages = navigate_naive('United_States', 'Gold(III)_chloride', click_demonstrations, links['United_States'], conf='conf1', fewshot=15, backtrack=True, n=1)
prompt = "\n".join([msg['content'] for msg in messages])
print(prompt)
print(f"\nNumber of tokens in prompt: {len(tokenizer.encode(prompt))}")



You are a knowledge discovery expert familiar with the Wikipedia link structure and your objective is to play the game of Wikispeedia: https://dlab.epfl.ch/wikispeedia/play/.
##Goal 
Given two Wikipedia articles, a source and a target, your goal is to reach the target article starting from the source article in as few clicks as possible. For the articles you are given this is always possible.

        ##Constraint 
You should exclusively follow the links present in the articles that you encounter along the way.

        ##Fine-grained instructions 
1. While the overall goal is to find a path from a source to a target article, you will proceed step by step.
2. Given outgoing links from the source article as candidates, you should select the candidate that takes you closer to the target article. Use your knowledge of the "expected" Wikipedia link structure and relatedness between articles to identify the candidate that takes you closer to the target.
3. If none of the provided candidates

In [30]:
def extract_json_from_response(response):
    '''
    test_str = ("This is a funny text about stuff,\n"
    "look at this product {\"action\":\"product\",\"options\":{\"action\":\"product\", \"action\":\"product\"}}.\n"
    "More Text is to come and another JSON string\n"
    "{\"action\":\"review\",\"options\":{...}}")
    matches = regex.finditer(pattern, test_str, regex.VERBOSE)
    for matchNum, match in enumerate(matches, start=1):

    print ("Match {matchNum} was found at {start}-{end}: {match}".format(matchNum = matchNum, start = match.start(), end = match.end(), match = match.group()))

    for groupNum in range(0, len(match.groups())):
        groupNum = groupNum + 1

        print ("Group {groupNum} found at {start}-{end}: {group}".format(groupNum = groupNum, start = match.start(groupNum), end = match.end(groupNum), group = match.group(groupNum)))
    '''
    pattern = r"""\{(?:[^{}]|(?R))*\}"""
    matches = regex.finditer(pattern, response, regex.VERBOSE)

    return [match.group() for matchNum, match in enumerate(matches, start=1)]

In [31]:
def gen_next_click(src, tgt, click_demonstrations, cands, conf, fewshot=5, backtrack=True):
    total_tokens_used =0; num_tries = 0
    while(num_tries<3):
        try:
            response, tokens_used = navigate_naive(src, tgt, click_demonstrations, cands, conf, fewshot, backtrack)
            total_tokens_used += tokens_used
            num_tries+=1
            #print("RAW Response", response)
            response_json_obj = extract_json_from_response(response[0])
            #print("JSON extracted from response", response_json_obj)
            if response_json_obj:
                gen_click_obj = json.loads(response_json_obj[0])
                if 'Output' in gen_click_obj:
                    gen_click = gen_click_obj['Output'].split(";")[0]
                else:
                    gen_click="NA"
                    print(gen_click_obj)
            else:
                gen_click="NA"
                print(response_json_obj)
            if gen_click not in cands:
                gen_click='NA'
            if gen_click!='NA':
                break
        except (openai.error.RateLimitError, openai.error.APIError, openai.error.Timeout, openai.error.ServiceUnavailableError) as e1:
            print(f'Encountered an exception {e1}, sleeping')
            print(num_tries)
            sleep(10)
            gen_click='NA'
        except ValueError as e2:
            print(f'Encountered an exception {e2}')
            print(response[0])
            print(num_tries)
            gen_click='NA'
    return gen_click, total_tokens_used

In [32]:
def navigate_wrapper(src, tgt, gen_path, gen_path_annotated, click_demonstrations, links, cands, total_tokens_used, conf, fewshot=5, backtrack=True):
    if tgt in links[src] or len(gen_path)>20:
        return gen_path, gen_path_annotated, total_tokens_used
    gen_click, tokens_used = gen_next_click(src, tgt, click_demonstrations, cands, conf, fewshot, backtrack)
    # print(src, tgt, gen_click, gen_path, gen_path_annotated, tokens_used)
    # print(src, tgt, gen_click, tokens_used)
    if not backtrack:
        if gen_click == 'NA':
            return gen_path+['NA'], gen_path_annotated+['NA'], total_tokens_used+tokens_used
        else:
            return navigate_wrapper(gen_click, tgt, gen_path+[gen_click], gen_path_annotated+[gen_click], click_demonstrations, links, links[gen_click], total_tokens_used+tokens_used, conf, fewshot, backtrack)
    else:
        if gen_click == 'NA' and len(gen_path)<=1:
            return gen_path+["<"], gen_path_annotated+["<"], total_tokens_used+tokens_used
        elif gen_click == 'NA':
            new_cands = links[gen_path[-2]]
            if src in new_cands:
                new_cands.remove(src)
            else:
                print(f'In navigate_wrapper, source={src}, target={tgt}, next_click={gen_click}, {gen_path_annotated}, {tokens_used}')
            return navigate_wrapper(gen_path[-2], tgt, gen_path[:-1], gen_path_annotated+['<'], click_demonstrations, links, new_cands, total_tokens_used+tokens_used, conf, fewshot, backtrack)
        else:
            return navigate_wrapper(gen_click, tgt, gen_path+[gen_click], gen_path_annotated+[gen_click], click_demonstrations, links, links[gen_click], total_tokens_used+tokens_used, conf, fewshot, backtrack)

In [163]:
# for backtrack in [False, True]:
for backtrack in [False]:
    print(f'Backtracking={backtrack}')
    # for fewshot in [0,1,5,10,15,20]:
    for fewshot in [0, 10]:
        print(f'Fewshot demonstrations={fewshot}')
        paths_generated = []
        for index, row in paths_real_finished_sample.iterrows():
    #     paths_generated = paths_generated[:295]
    #     for index, row in paths_real_finished_sample.iloc[59:].iterrows():
            print(row['src'], row['tgt'], row['path_len'])
            for conf in ['conf1', 'conf2', 'conf3', 'conf4', 'conf5']:
                attempt = 0
    #             for i in range(2):
                attempt+=1
                links_dict = copy.deepcopy(links)
                gen_path, gen_path_annotated, num_tokens = navigate_wrapper(row['src'], row['tgt'], [row['src']], [row['src']], click_demonstrations, links_dict, links_dict[row['src']], 0, conf, fewshot, backtrack)
                if not backtrack:
                    if gen_path_annotated[-1] == "NA":
                        print(gen_path_annotated, row['tgt'], 'Unsuccessful', num_tokens, attempt, payload[conf]['temperature'], fewshot)
                        paths_generated.append([row['src'], row['tgt'], row['path'], row['path_len'], gen_path_annotated, len(gen_path_annotated), 0, num_tokens, num_tokens*0.002/1000, attempt, payload[conf]['temperature'], fewshot])
                    elif row['tgt'] in links[gen_path_annotated[-1]]:
                        print(gen_path_annotated+[row['tgt']], 'Successful', num_tokens, attempt, payload[conf]['temperature'], fewshot)
                        paths_generated.append([row['src'], row['tgt'], row['path'], row['path_len'], gen_path_annotated+[row['tgt']], len(gen_path_annotated), 1, num_tokens, num_tokens*0.002/1000, attempt, payload[conf]['temperature'], fewshot])
                    else:
                        print(gen_path_annotated, row['tgt'], 'Unsuccessful', num_tokens, attempt, payload[conf]['temperature'], fewshot)
                        paths_generated.append([row['src'], row['tgt'], row['path'], row['path_len'], gen_path_annotated, len(gen_path_annotated), 0, num_tokens, num_tokens*0.002/1000, attempt, payload[conf]['temperature'], fewshot])
                else:
                    if gen_path_annotated[-1] == "<":
                        print(gen_path_annotated, row['tgt'], 'Unsuccessful', num_tokens, attempt, payload[conf]['temperature'], fewshot)
                        paths_generated.append([row['src'], row['tgt'], row['path'], row['path_len'], gen_path_annotated, len(gen_path_annotated), 0, num_tokens, num_tokens*0.002/1000, attempt, payload[conf]['temperature'], fewshot])
                    elif row['tgt'] in links[gen_path_annotated[-1]]:
                        print(gen_path_annotated+[row['tgt']], 'Successful', num_tokens, attempt, payload[conf]['temperature'], fewshot)
                        paths_generated.append([row['src'], row['tgt'], row['path'], row['path_len'], gen_path_annotated+[row['tgt']], len(gen_path_annotated), 1, num_tokens, num_tokens*0.002/1000, attempt, payload[conf]['temperature'], fewshot])
                    else:
                        print(gen_path_annotated, row['tgt'], 'Unsuccessful', num_tokens, attempt, payload[conf]['temperature'], fewshot)
                        paths_generated.append([row['src'], row['tgt'], row['path'], row['path_len'], gen_path_annotated, len(gen_path_annotated), 0, num_tokens, num_tokens*0.002/1000, attempt, payload[conf]['temperature'], fewshot])

        paths_generated_df = pd.DataFrame(paths_generated, columns=['src', 'tgt', 'human_path', 'human_path_len', 'generated_path', 'generated_path_len', 'is_finished', 'num_tokens', 'USD', 'attempt', 'temperature', 'num_demonstrations'])
        paths_generated_df.to_csv(f'8Aug_generated_paths_naive_backtrack_{backtrack}_demonstrations_{fewshot}.csv', index=False)

Backtracking=False
Fewshot demonstrations=0
Jurassic Cologne 2
['Jurassic', 'Germany', 'Cologne'] Successful 462 1 1 0
['Jurassic', 'Germany', 'Cologne'] Successful 462 1 0.75 0
['Jurassic', 'Germany', 'Cologne'] Successful 462 1 0.5 0
['Jurassic', 'Germany', 'Cologne'] Successful 462 1 0.25 0
['Jurassic', 'Germany', 'Cologne'] Successful 462 1 0.1 0
Brain Telephone 7
['Brain', 'Cell_(biology)', 'Protein', 'Cell_(biology)', 'Mitochondrion', 'Cell_(biology)', 'Energy', 'Communication', 'Telephone'] Successful 3743 1 1 0
['Brain', 'Action_potential', 'Chemical_synapse', 'Action_potential', 'Electrical_resistance', 'Superconductivity', 'Alternating_current', 'Electricity', 'Alexander_Graham_Bell', 'Telephone'] Successful 3519 1 0.75 0
['Brain', 'Computer_science', 'Internet', 'World_Wide_Web', 'Telephone'] Successful 1432 1 0.5 0
['Brain', 'Cell_(biology)', 'Microscope', 'Optical_microscope', 'Microscope', 'Optical_microscope', 'Microscope', 'Optical_microscope', 'Microscope', 'Optical_mi

In [83]:
paths_generated_df = pd.read_csv('28Jul_generated_paths_naive_demonstrations_0.csv')
paths_generated_df.groupby('temperature').agg({'is_finished': ['sum', 'mean'], 'generated_path_len': ['median', 'mean']})

is_finished       generated_path_len      
                    sum  mean             median  mean
temperature                                           
0.10                 61  0.61                4.5  9.26
0.25                 66  0.66                5.0  8.61
0.50                 71  0.71                6.0  9.02
0.75                 74  0.74                6.0  9.62
1.00                 83  0.83                6.0  8.57

In [115]:
paths_generated_df = pd.read_csv('28Jul_generated_paths_naive_demonstrations_10.csv')
paths_generated_df.groupby('temperature').agg({'is_finished': ['sum', 'mean'], 'generated_path_len': ['median', 'mean']})

is_finished       generated_path_len      
                    sum  mean             median  mean
temperature                                           
0.10                 77  0.77                4.0  7.80
0.25                 76  0.76                5.0  8.94
0.50                 81  0.81                5.0  8.16
0.75                 86  0.86                5.0  7.90
1.00                 90  0.90                5.0  7.38

In [134]:
paths_generated_memory_df = pd.read_csv('28Jul_generated_paths_memory_demonstrations_0.csv')
paths_generated_memory_df.groupby('temperature').agg({'is_finished': ['sum', 'mean'], 'generated_path_len': ['median', 'mean']})

is_finished       generated_path_len       
                    sum  mean             median   mean
temperature                                            
0.10                 37  0.37                4.5  11.43
0.25                 43  0.43                5.0  11.66
0.50                 49  0.49                6.0  11.58
0.75                 63  0.63                6.0  11.15
1.00                 69  0.69                8.0  11.76

In [135]:
paths_generated_df = pd.read_csv('28Jul_generated_paths_memory_demonstrations_10.csv')
paths_generated_df.groupby('temperature').agg({'is_finished': ['sum', 'mean'], 'generated_path_len': ['median', 'mean']})

is_finished       generated_path_len      
                    sum  mean             median  mean
temperature                                           
0.10                 59  0.59                4.0  8.71
0.25                 63  0.63                4.0  8.68
0.50                 66  0.66                4.0  9.07
0.75                 71  0.71                5.0  9.33
1.00                 73  0.73                5.0  9.54

In [65]:
demonstrations = [{"role":"system", "content": 
             f'''You are a knowledge discovery expert familiar with the Wikipedia link structure and your objective is to play the game of Wikispeedia: https://dlab.epfl.ch/wikispeedia/play/.'''}]

demonstrations.append({"role": "user", "content": '''\n##Goal \nGiven two Wikipedia articles, a source and a target, your goal is to reach the target article starting from the source article in as few clicks as possible. For the articles you are given this is always possible.\n
##Constraint \nYou should exclusively follow the links present in the articles that you encounter along the way.\n
##Fine-grained instructions \n1. While the overall goal is to find a path from a source to a target article, you will proceed step by step.\n2. Given outgoing links from the source article as candidates, you should select the candidate that takes you closer to the target article. Use your knowledge of the "expected" Wikipedia link structure and relatedness between articles to identify the candidate that takes you closer to the target.\n3. If none of the provided candidates takes you closer to the target article, select "NA".\n4. Choose **only** from the provided candidates.\n5. Do not provide an algorithm or code to solve the task, instead present only the solution.\n
##Input \nYou will be given input in the JSON format as described below\n
Input format:
{
    "Source": "<source>", 
    "Candidates": "<candidates>", 
    "Target": "<target>"
}

##Output \nYou should only respond in the JSON format as described below\n
Output format:
{
    "Output": "<output>"
}
Ensure the response can be parsed by Python json.loads
'''})

for entry in demonstrations:
    print('role:', entry['role'])
    print('content:', entry['content'])
    print()

click_demos = []
for demo in click_demonstrations[0:fewshot]:
    click_demos.append({"role": "user", "content": f'''{{"Source": "{demo['src']}", "Candidates": "{demo['cands']}", "Target": "{demo['tgt']}"}}'''})
    click_demos.append({"role": "assistant", "content": f'''{{"Output": "{demo['output']}"}}'''})
    
for entry in click_demos:
    print('role:', entry['role'])
    content_dict = json.loads(entry['content'])
    for key in content_dict:
        print(f'{key}: {content_dict[key]}')
    print()

role: system
content: You are a knowledge discovery expert familiar with the Wikipedia link structure and your objective is to play the game of Wikispeedia: https://dlab.epfl.ch/wikispeedia/play/.

role: user
content: 
##Goal 
Given two Wikipedia articles, a source and a target, your goal is to reach the target article starting from the source article in as few clicks as possible. For the articles you are given this is always possible.

##Constraint 
You should exclusively follow the links present in the articles that you encounter along the way.

##Fine-grained instructions 
1. While the overall goal is to find a path from a source to a target article, you will proceed step by step.
2. Given outgoing links from the source article as candidates, you should select the candidate that takes you closer to the target article. Use your knowledge of the "expected" Wikipedia link structure and relatedness between articles to identify the candidate that takes you closer to the target.
3. If non

## Cycle avoidance

In [33]:
def navigate_naive_no_cycles(src, tgt, click_demonstrations, cands, conf='conf1', fewshot=5, n=1):
    # Current system prompt
    demonstrations = [{"role":"system", "content": 
                 f'''You are a knowledge discovery expert familiar with the Wikipedia link structure and your objective is to play the game of Wikispeedia: https://dlab.epfl.ch/wikispeedia/play/.'''}]

    demonstrations.append({"role": "user", "content": '''##Goal \nGiven two Wikipedia articles, a source and a target, your goal is to reach the target article starting from the source article in as few clicks as possible. For the articles you are given this is always possible.\n
    ##Constraint \nYou should exclusively follow the links present in the articles that you encounter along the way.\n
    ##Fine-grained instructions \n1. While the overall goal is to find a path from a source to a target article, you will proceed step by step.\n2. Given outgoing links from the source article as candidates, you should select the candidate that takes you closer to the target article. Use your knowledge of the "expected" Wikipedia link structure and relatedness between articles to identify the candidate that takes you closer to the target.\n3. If none of the provided candidates takes you closer to the target article, select "NA".\n4. Choose **only** from the provided candidates.\n5. Do not provide an algorithm or code to solve the task, instead present only the solution.\n
    ##Input \nYou will be given input in the JSON format as described below\n
    Input format:
    {
        "Source": "<source>", 
        "Candidates": "<candidates>", 
        "Target": "<target>"
    }

    ##Output \nYou should only respond in the JSON format as described below\n
    Output format:
    {
        "Output": "<output>"
    }
    Ensure the response can be parsed by Python json.loads
    '''})

    # Include 5 few-shot demonstrations for ensuring that the model understands the input and output structure
    if fewshot:
        # always the same demonstrations
        for demo in click_demonstrations[0:fewshot]:
            demonstrations.append({"role": "user", "content": f'''{{"Source": "{demo['src']}", "Candidates": "{demo['cands']}", "Target": "{demo['tgt']}"}}'''})
            demonstrations.append({"role": "assistant", "content": f'''{{"Output": "{demo['output']}"}}'''})

    messages = []
    messages.extend(demonstrations)
    messages.append({"role": "user", "content": f'''{{"Source": "{src}", "Candidates": "{";".join(cands)}", "Target": "{tgt}"}}'''})
#     print(messages[-1])
#     print()

    # setting n very high is almost free.
    # the model only generates 2-3 tokens for each reply and we only pay for the demonstrations once
    # n=128 is the highest the API supports
    if fewshot>=20:
        response = openai.ChatCompletion.create(model="gpt-3.5-turbo-16k", **payload[conf], messages=messages, n=n)
    else:
        response = openai.ChatCompletion.create(model="gpt-3.5-turbo", **payload[conf], messages=messages, n=n)

    # # parsing the guesses
    response_text = [c['message']['content'] for c in response['choices']]
    return response_text, response.usage.total_tokens

In [35]:
def gen_next_click_no_cycles(src, tgt, click_demonstrations, cands, conf, fewshot=5):
    total_tokens_used =0; num_tries = 0
    while(num_tries<3):
        try:
            response, tokens_used = navigate_naive_no_cycles(src, tgt, click_demonstrations, cands, conf, fewshot)
            total_tokens_used += tokens_used
            num_tries+=1
            #print("RAW Response", response)
            response_json_obj = extract_json_from_response(response[0])
            #print("JSON extracted from response", response_json_obj)
            if response_json_obj:
                gen_click_obj = json.loads(response_json_obj[0])
                if 'Output' in gen_click_obj:
                    if isinstance(gen_click_obj['Output'], str):
                        gen_click = gen_click_obj['Output'].split(";")[0]
                    elif isinstance(gen_click_obj['Output'], list):
                        gen_click = gen_click_obj['Output'][0]
                    else:
                        print(f'Ignoring un-parsable object {gen_click_obj}, gen_click=NA')
                        gen_click = "NA"
                else:
                    gen_click="NA"
                    print(gen_click_obj)
            else:
                gen_click="NA"
                print(response_json_obj)
            if gen_click not in cands:
                gen_click='NA'
            if gen_click!='NA':
                break
        except (openai.error.RateLimitError, openai.error.APIError, openai.error.Timeout, openai.error.ServiceUnavailableError) as e1:
            print(f'Encountered an exception {e1}, sleeping')
            print(num_tries)
            sleep(10)
            gen_click='NA'
        except ValueError as e2:
            print(f'Encountered an exception {e2}')
            print(response[0])
            print(num_tries)
            gen_click='NA'
    return gen_click, total_tokens_used

In [38]:
def navigate_wrapper_no_cycles(src, tgt, gen_path, gen_path_annotated, visited, click_demonstrations, links, cands, total_tokens_used, conf, fewshot=5):
    if tgt in links[src] or len(gen_path)>20:
        return gen_path, gen_path_annotated, total_tokens_used
    effective_cands = list(set(cands) - visited)
    gen_click, tokens_used = gen_next_click_no_cycles(src, tgt, click_demonstrations, effective_cands, conf, fewshot)
    # print(src, tgt, gen_click, gen_path, gen_path_annotated, tokens_used)
    # print(src, tgt, gen_click, tokens_used)
    if gen_click == 'NA' and len(gen_path)<=1:
        return gen_path+["<"], gen_path_annotated+["<"], total_tokens_used+tokens_used
    elif gen_click == 'NA':
        new_cands = links[gen_path[-2]]
        if src in new_cands:
            new_cands.remove(src)
        else:
            print(f'In navigate_wrapper, source={src}, target={tgt}, next_click={gen_click}, {gen_path_annotated}, {tokens_used}')
        return navigate_wrapper_no_cycles(gen_path[-2], tgt, gen_path[:-1], gen_path_annotated+['<'], visited, click_demonstrations, links, new_cands, total_tokens_used+tokens_used, conf, fewshot)
    else:
        visited.add(gen_click)
        return navigate_wrapper_no_cycles(gen_click, tgt, gen_path+[gen_click], gen_path_annotated+[gen_click], visited, click_demonstrations, links, links[gen_click], total_tokens_used+tokens_used, conf, fewshot)

In [41]:
# for fewshot in [0,1,5,10,15,20]:
for fewshot in [0, 10]:
    print(f'Fewshot demonstrations={fewshot}')
    paths_generated = []
    for index, row in paths_real_finished_sample.iterrows():
#     paths_generated = paths_generated[:295]
#     for index, row in paths_real_finished_sample.iloc[59:].iterrows():
        print(row['src'], row['tgt'], row['path_len'])
        for conf in ['conf1', 'conf2', 'conf3', 'conf4', 'conf5']:
            attempt = 0
#             for i in range(2):
            attempt+=1
            links_dict = copy.deepcopy(links)
            gen_path, gen_path_annotated, num_tokens = navigate_wrapper_no_cycles(row['src'], row['tgt'], [row['src']], [row['src']], set(), click_demonstrations, links_dict, links_dict[row['src']], 0, conf, fewshot)
            if gen_path_annotated[-1] == "<":
                print(gen_path_annotated, row['tgt'], 'Unsuccessful', num_tokens, attempt, payload[conf]['temperature'], fewshot)
                paths_generated.append([row['src'], row['tgt'], row['path'], row['path_len'], gen_path_annotated, len(gen_path_annotated), 0, num_tokens, num_tokens*0.002/1000, attempt, payload[conf]['temperature'], fewshot])
            elif row['tgt'] in links[gen_path_annotated[-1]]:
                print(gen_path_annotated+[row['tgt']], 'Successful', num_tokens, attempt, payload[conf]['temperature'], fewshot)
                paths_generated.append([row['src'], row['tgt'], row['path'], row['path_len'], gen_path_annotated+[row['tgt']], len(gen_path_annotated), 1, num_tokens, num_tokens*0.002/1000, attempt, payload[conf]['temperature'], fewshot])
            else:
                print(gen_path_annotated, row['tgt'], 'Unsuccessful', num_tokens, attempt, payload[conf]['temperature'], fewshot)
                paths_generated.append([row['src'], row['tgt'], row['path'], row['path_len'], gen_path_annotated, len(gen_path_annotated), 0, num_tokens, num_tokens*0.002/1000, attempt, payload[conf]['temperature'], fewshot])

    paths_generated_df = pd.DataFrame(paths_generated, columns=['src', 'tgt', 'human_path', 'human_path_len', 'generated_path', 'generated_path_len', 'is_finished', 'num_tokens', 'USD', 'attempt', 'temperature', 'num_demonstrations'])
    paths_generated_df.to_csv(f'3Sep_generated_paths_naive_no_cycles_demonstrations_{fewshot}.csv', index=False)

Fewshot demonstrations=0
Jurassic Cologne 2
['Jurassic', 'Germany', 'Cologne'] Successful 482 1 1 0
['Jurassic', 'Germany', 'Cologne'] Successful 482 1 0.75 0
['Jurassic', 'Germany', 'Cologne'] Successful 482 1 0.5 0
['Jurassic', 'Germany', 'Cologne'] Successful 482 1 0.25 0
['Jurassic', 'Germany', 'Cologne'] Successful 482 1 0.1 0
Brain Telephone 7
['Brain', 'Cell_(biology)', 'James_D._Watson', 'Seattle,_Washington', 'Telephone'] Successful 1483 1 1 0
['Brain', '<'] Telephone Unsuccessful 1740 1 0.75 0
['Brain', '<'] Telephone Unsuccessful 1740 1 0.5 0
['Brain', '<'] Telephone Unsuccessful 1740 1 0.25 0
['Brain', '<'] Telephone Unsuccessful 1740 1 0.1 0
Honey September_11,_2001_attacks 2
['Honey', 'United_States', 'September_11,_2001_attacks'] Successful 499 1 1 0
['Honey', 'United_States', 'September_11,_2001_attacks'] Successful 499 1 0.75 0
['Honey', 'United_States', 'September_11,_2001_attacks'] Successful 499 1 0.5 0
['Honey', 'United_States', 'September_11,_2001_attacks'] Succes

## Adding Memory

In [42]:
def navigate_with_memory(src, cur, tgt, path_so_far, click_demonstrations, cands, conf='conf1', fewshot=5, n=1):    
    # System prompt
    demonstrations = [{"role":"system", "content": 
                 f'''You are a knowledge discovery expert familiar with the Wikipedia link structure and your objective is to play the game of Wikispeedia: https://dlab.epfl.ch/wikispeedia/play/.'''}]

    # Initial prompt for describing the task
    ##Performance Evaluation \n1. The shorter the path the better, continuously track the number of articles visited in the "traversed path" part of input.\n2. Analyze the "traversed_path" part of input to reflect on past decisions and strategies to refine your approach.\n
    demonstrations.append({"role": "user", "content": '''##Goal \nGiven two Wikipedia articles, a source and a target, your goal is to reach the target article starting from the source article in as few clicks as possible. For the articles you are given this is always possible.\n
    ##Constraint \nYou should exclusively follow the links present in the articles that you encounter along the way.\n
    ##Fine-grained instructions \n1. While the overall goal is to find a path from a source to a target article, you will proceed step by step.\n2. Given outgoing links from the current article as candidates and the list of articles that you have visited so far, you should select the candidate that takes you closer to the target article. Use your knowledge of the "expected" Wikipedia link structure and relatedness between articles to identify the candidate that takes you closer to the target.\n3. If none of the provided candidates takes you closer to the target article, select "NA".\n4. Choose **only** from the provided candidates.\n5. Do not provide an algorithm or code to solve the task, instead present only the solution.\n
    ##Performance Evaluation \nAnalyze the "Traversed path" field of input JSON to reflect on past decisions and strategies to refine your approach.\n
    ##Input \nYou will be given input in the JSON format as described below\n
    Input format:
    {
        "Source": "<source article>", 
        "Traversed path": "<articles visited so far>", 
        "Current": "<current article>", 
        "Candidates": "<outgoing links from current article>", 
        "Target": "<target article>"
    }

    ##Output \nYou should only respond in the JSON format as described below\n
    Output format:
    {
        "Output": "<output>"
    }
    Ensure the response can be parsed by Python json.loads
    '''})

    # Include 5 few-shot demonstrations for ensuring that the model understands the input and output structure.
    if fewshot:
        # always the same demonstrations
        for demo in click_demonstrations[0:fewshot]:
            demonstrations.append({"role": "user", "content": f'''{{"Source": "{demo['src']}", "Traversed path": "{demo['path_so_far']}", "Current": "{demo['cur']}", "Candidates": "{demo['cands']}", "Target": "{demo['tgt']}"}}'''})
            demonstrations.append({"role": "assistant", "content": f'''{{"Output": "{demo['output']}"}}'''})

    messages = []
    messages.extend(demonstrations)
    # messages.append({"role": "user", "content": f'''{{"Source": "{src}", "Traversed path": "{"->".join(path_so_far)}", "Current": "{cur}", "Candidates": "{",".join(cands)}", "Target": "{tgt}"}}'''})
    messages.append({"role": "user", "content": f'''{{"Source": "{src}", "Traversed path": "{";".join(path_so_far)}", "Current": "{cur}", "Candidates": "{";".join(cands)}", "Target": "{tgt}"}}'''})
#     print("In navigate_with_memory:", messages[-1])
#     print()

    # setting n very high is almost free.
    # the model only generates 2-3 tokens for each reply and we only pay for the demonstrations once
    # n=128 is the highest the API supports
    if fewshot>=10:
        response = openai.ChatCompletion.create(model="gpt-3.5-turbo-16k", **payload[conf], messages=messages, n=n)
    else:
        response = openai.ChatCompletion.create(model="gpt-3.5-turbo", **payload[conf], messages=messages, n=n)

    # # parsing the guesses
    response_text = [c['message']['content'] for c in response['choices']]
    return response_text, response.usage.total_tokens

In [43]:
def gen_next_click_with_memory(src, cur, tgt, path_so_far, click_demonstrations, cands, conf, fewshot=5):
    total_tokens_used =0; num_tries = 0
    # print(f'In gen_next_click_with_memory, source={src}, current={cur}, target={tgt}, path_so_far={path_so_far}')
    while(num_tries<3):
        try:
            response, tokens_used = navigate_with_memory(src, cur, tgt, path_so_far, click_demonstrations, cands, conf, fewshot)
            total_tokens_used += tokens_used
            num_tries+=1
            response_json_obj = extract_json_from_response(response[0])
#             print(f'In gen_next_click_with_memory, num_tries={num_tries}, cur={cur}, cands={cands}, response={response_json_obj}')
            if response_json_obj:
                gen_click_obj = json.loads(response_json_obj[0])
                if 'Output' in gen_click_obj:
                    if isinstance(gen_click_obj['Output'], str):
                        gen_click = gen_click_obj['Output'].split(";")[0]
                    elif isinstance(gen_click_obj['Output'], list):
                        gen_click = gen_click_obj['Output'][0]
                    else:
                        print(f'Ignoring un-parsable object {gen_click_obj}, gen_click=NA')
                        gen_click = "NA"
                else:
                    gen_click="NA"
                    print(gen_click_obj)
            else:
                gen_click="NA"
                print(response_json_obj)
            if gen_click not in cands:
                gen_click='NA'
            if gen_click!='NA':
                break
        except (openai.error.RateLimitError, openai.error.APIError, openai.error.Timeout, openai.error.ServiceUnavailableError) as e1:
            print(f'Encountered an exception {e1}, sleeping')
            print(num_tries)
            gen_click='NA'
            sleep(10)
        except ValueError as e2:
            print(f'Encountered an exception {e2}')
            print(response[0])
            print(num_tries)
            gen_click='NA'
    return gen_click, total_tokens_used

In [44]:
def navigate_wrapper_with_memory(src, cur, tgt, path_so_far, gen_path, gen_path_annotated, visited, click_demonstrations, links, cands, total_tokens_used, conf, fewshot=5):
    if tgt in links[cur] or len(gen_path)>20:
        return gen_path, gen_path_annotated, total_tokens_used
    effective_cands = list(set(cands) - visited)
    gen_click, tokens_used = gen_next_click_with_memory(src, cur, tgt, path_so_far, click_demonstrations, effective_cands, conf, fewshot)
    # print(f'In navigate_wrapper_with_memory, source={src}, current={cur}, target={tgt}, next_click={gen_click}, path_so_far={path_so_far}, {gen_path_annotated}, {tokens_used}')
    # print(src, tgt, gen_click, tokens_used)
    if gen_click == 'NA' and len(gen_path)<=1:
        return gen_path+["<"], gen_path_annotated+["<"], total_tokens_used+tokens_used
    elif gen_click == 'NA':
        new_cands = links[gen_path[-2]]
        # print(gen_path[-2], new_cands)
        if cur in new_cands:
            new_cands.remove(cur)
        else:
            print(f'In navigate_wrapper_with_memory, source={src}, current={cur}, target={tgt}, next_click={gen_click}, path_so_far={path_so_far}, {gen_path_annotated}, {tokens_used}')
        return navigate_wrapper_with_memory(src, gen_path[-2], tgt, path_so_far+[gen_path[-2]], gen_path[:-1], gen_path_annotated+['<'], visited, click_demonstrations, links, new_cands, total_tokens_used+tokens_used, conf, fewshot)
    else:
        visited.add(gen_click)
        return navigate_wrapper_with_memory(src, gen_click, tgt, path_so_far+[gen_click], gen_path+[gen_click], gen_path_annotated+[gen_click], visited, click_demonstrations, links, links[gen_click], total_tokens_used+tokens_used, conf, fewshot)

In [45]:
# for fewshot in [0,1,5,10,15,20]:
for fewshot in [0,10]:
    print(f'Fewshot demonstrations={fewshot}')
    if fewshot==25:
        start_row=42
        paths_generated_memory=paths_generated_memory[:168]
    else:
        start_row=0
        paths_generated_memory = []
    for index, row in paths_real_finished_sample.iloc[start_row:].iterrows():
        print(row['src'], row['tgt'], row['path_len'])
        for conf in ['conf1', 'conf2', 'conf3', 'conf4', 'conf5']:
            attempt = 0
            attempt+=1
            links_dict = copy.deepcopy(links)
            gen_path, gen_path_annotated, num_tokens = navigate_wrapper_with_memory(row['src'], row['src'], row['tgt'], [row['src']], [row['src']], [row['src']], set(), click_demonstrations_with_memory, links_dict, links_dict[row['src']], 0, conf, fewshot)
            if gen_path_annotated[-1] == "<":
                print(gen_path_annotated, row['tgt'], 'Unsuccessful', num_tokens, attempt, payload[conf]['temperature'], fewshot)
                paths_generated_memory.append([row['src'], row['tgt'], row['path'], row['path_len'], gen_path_annotated, len(gen_path_annotated), 0, num_tokens, num_tokens*0.002/1000, attempt, payload[conf]['temperature'], fewshot])
            elif row['tgt'] in links[gen_path_annotated[-1]]:
                print(gen_path_annotated+[row['tgt']], 'Successful', num_tokens, attempt, payload[conf]['temperature'], fewshot)
                paths_generated_memory.append([row['src'], row['tgt'], row['path'], row['path_len'], gen_path_annotated+[row['tgt']], len(gen_path_annotated), 1, num_tokens, num_tokens*0.002/1000, attempt, payload[conf]['temperature'], fewshot])
            else:
                print(gen_path_annotated, row['tgt'], 'Unsuccessful', num_tokens, attempt, payload[conf]['temperature'], fewshot)
                paths_generated_memory.append([row['src'], row['tgt'], row['path'], row['path_len'], gen_path_annotated, len(gen_path_annotated), 0, num_tokens, num_tokens*0.002/1000, attempt, payload[conf]['temperature'], fewshot])

    paths_generated_mem_df = pd.DataFrame(paths_generated_memory, columns=['src', 'tgt', 'human_path', 'human_path_len', 'generated_path', 'generated_path_len', 'is_finished', 'num_tokens', 'USD', 'attempt', 'temperature', 'num_demonstrations'])
    paths_generated_mem_df.to_csv(f'3Sep_generated_paths_memory_no_cycles_demonstrations_{fewshot}.csv', index=False)

Fewshot demonstrations=0
Jurassic Cologne 2
['Jurassic', 'Germany', 'Cologne'] Successful 573 1 1 0
['Jurassic', 'Germany', 'Cologne'] Successful 570 1 0.75 0
['Jurassic', 'Germany', 'Cologne'] Successful 573 1 0.5 0
['Jurassic', 'Germany', 'Cologne'] Successful 570 1 0.25 0
['Jurassic', 'Germany', 'Cologne'] Successful 570 1 0.1 0
Brain Telephone 7
Encountered an exception Request timed out: HTTPSConnectionPool(host='api.openai.com', port=443): Read timed out. (read timeout=30), sleeping
1
['Brain', '<'] Telephone Unsuccessful 1998 1 1 0
['Brain', "Alzheimer's_disease", 'DNA', 'Computer_science', 'Internet', '<', 'Information', 'Communication', 'Telephone'] Successful 5602 1 0.75 0
Encountered an exception Request timed out: HTTPSConnectionPool(host='api.openai.com', port=443): Read timed out. (read timeout=30), sleeping
2
['Brain', '<'] Telephone Unsuccessful 1998 1 0.5 0
['Brain', '<'] Telephone Unsuccessful 1998 1 0.25 0
['Brain', '<'] Telephone Unsuccessful 1998 1 0.1 0
Honey Sept

In [465]:
def navigate_with_strategies(src, cur, tgt, path_so_far, click_demonstrations, cands, conf='conf1', fewshot=5, n=1):
    # System prompt
    demonstrations = [{"role":"system", "content": 
                 f'''You are a knowledge discovery expert familiar with the Wikipedia link structure and your objective is to play the game of Wikispeedia: https://dlab.epfl.ch/wikispeedia/play/.'''}]

    # Initial prompt for describing the task
    ##Performance Evaluation \n1. The shorter the path the better, continuously track the number of articles visited in the "traversed path" part of input.\n2. Analyze the "traversed_path" part of input to reflect on past decisions and strategies to refine your approach.\n
    demonstrations.append({"role": "user", "content": '''##Goal \nGiven two Wikipedia articles, a source and a target, your goal is to reach the target article starting from the source article. For the articles you are given this is always possible.\n
    ##Constraint \nYou should exclusively follow the links present in the articles that you encounter along the way.\n
    ##Fine-grained instructions \n1. While the overall goal is to find a path from a source to a target article, you will proceed step by step.\n2. Given outgoing links from the current article as candidates, you should select the candidate that takes you closer to the target article. Use your knowledge of the "expected" Wikipedia link structure and relatedness between articles to identify the candidate that takes you closer to the target.\n3. If none of the provided candidates takes you closer to the target article, select "NA".\n4. Choose **only** from the provided candidates.\n5. Do not provide an algorithm or code to solve the task, instead present only the solution.\n
    ##Performance Evaluation \nAnalyze the "Traversed path" field of input JSON to reflect on past decisions and strategies to refine your approach.\n
    ##Input \nYou will be given input in the JSON format as described below\n
    Input format:
    {
        "Source": "<source article>", 
        "Traversed path": "<articles visited so far>", 
        "Current": "<current article>", 
        "Candidates": "<outgoing links from current article>", 
        "Target": "<target article>"
    }

    ##Output \nYou should only respond in the JSON format as described below\n
    Output format:
    {
        "thoughts": {
            "reasoning": "<reasoning behind the chosen candidate>",
            "plan": "<- short bulleted\n- list that conveys\n- long-term plan to reach the target>"
        }
        "Output": "<output>", 
    }
    Ensure the response can be parsed by Python json.loads
    '''})

    # Include 5 few-shot demonstrations for ensuring that the model understands the input and output structure.
    if fewshot:
        # always the same demonstrations
        for demo in click_demonstrations[0:fewshot]:
            demonstrations.append({"role": "user", "content": f'''{{"Source": "{demo['src']}", "Traversed path": "{demo['path_so_far']}", "Current": "{demo['cur']}", "Candidates": "{demo['cands']}", "Target": "{demo['tgt']}"}}'''})
            demonstrations.append({"role": "assistant", "content": f'''{{"Output": "{demo['output']}"}}'''})

    messages = []
    messages.extend(demonstrations)
    # messages.append({"role": "user", "content": f'''{{"Source": "{src}", "Traversed path": "{"->".join(path_so_far)}", "Current": "{cur}", "Candidates": "{",".join(cands)}", "Target": "{tgt}"}}'''})
    messages.append({"role": "user", "content": f'''{{"Source": "{src}", "Traversed path": "{";".join(path_so_far)}", "Current": "{cur}", "Candidates": "{";".join(cands)}", "Target": "{tgt}"}}'''})
    print("In navigate_with_memory:", messages[-1])
    print()

    # setting n very high is almost free.
    # the model only generates 2-3 tokens for each reply and we only pay for the demonstrations once
    # n=128 is the highest the API supports
    if fewshot>=10:
        response = openai.ChatCompletion.create(model="gpt-3.5-turbo-16k", **payload[conf], messages=messages, n=n)
    else:
        response = openai.ChatCompletion.create(model="gpt-3.5-turbo", **payload[conf], messages=messages, n=n)

    # # parsing the guesses
    response_text = [c['message']['content'] for c in response['choices']]
    return response_text, response.usage.total_tokens

In [471]:
i = 0
for index, row in paths_real_finished_sample.iterrows():
    if i==5:
        break
    print(row['src'], row['tgt'], row['path_len'])
    for conf in ['conf_test1', 'conf_test2']:
        response, tokens_used = navigate_plan(row['src'], row['tgt'], conf)
        print(payload[conf])
        print("RAW Response", response)
        response_json_obj = extract_json_from_response(response[0])
        print("JSON extracted from response", response_json_obj)
        try:
            gen_click_obj = json.loads(response[0])['Output']
        except ValueError as e:
            print(e, gen_click_obj)
        print(tokens_used)
        print()
    i+=1

Jurassic Cologne 2
{'role': 'user', 'content': '{"Source": "Jurassic", "Target": "Cologne"}'}

{'max_tokens': 256, 'temperature': 1, 'top_p': 1}
RAW Response ['{\n    "Output": "Jurassic; Jurassic Park (film); Stanley Kubrick; Germany; Cologne",\n    \n    "thoughts": {\n        "strategy": \n            "- Explore links in the source article for related topics\\n- Follow the links in these related topics as far as needed to reach the target article",\n        \n        "reasoning": \n            "Jurassic Park always reminds me of Stanley Kubrick who was known for directing multiple genre films. One of Kubrick\'s top-grossing films was A Clockwork Orange which has a cultural impact on Germany. This can be confirmed by exploring the \'See also\' section of the Jurassic Park Wikipedia page and looking at the articles related to Germany, where Cologne is located."\n    }\n}']
JSON extracted from response ['{\n    "Output": "Jurassic; Jurassic Park (film); Stanley Kubrick; Germany; Cologn

In [32]:
i = 0
for index, row in paths_real_finished_sample.iterrows():
    if i==5:
        break
    print(row['src'], row['tgt'], row['path_len'])
    for conf in ['conf_test1', 'conf_test2']:
        response, tokens_used = navigate_naive(row['src'], row['tgt'], click_demonstrations, links[row['src']], conf, 0)
        print(payload[conf])
        print("RAW Response", response)
        response_json_obj = extract_json_from_response(response[0])
        print("JSON extracted from response", response_json_obj)
        try:
            gen_click_obj = json.loads(response[0])['Output']
            print(gen_click_obj, gen_click_obj.split(";")[0])
        except ValueError as e:
            print(e, gen_click_obj)
        print(tokens_used)
        print()
    i+=1

Jurassic Cologne 2
{'role': 'user', 'content': '{"Source": "Jurassic", "Candidates": "Ammonite;Atlantic_Ocean;Batholith;Canada;Cretaceous;Dinosaur;Europe;Evolution;Fern;Fish;France;Germany;Glacier;Hip_hop_music;Ichthyosaur;India;Invertebrate;Japan;North_America;Ornithischia;Pterosaur;Reptile;Russia;Saurischia;South_America;Switzerland;Triassic;United_Kingdom;United_States;World_Heritage_Site", "Target": "Cologne"}'}

{'max_tokens': 256, 'temperature': 1, 'top_p': 1}
RAW Response ['{"Output": "Germany"}']
JSON extracted from response ['{"Output": "Germany"}']
Germany Germany
476

{'role': 'user', 'content': '{"Source": "Jurassic", "Candidates": "Ammonite;Atlantic_Ocean;Batholith;Canada;Cretaceous;Dinosaur;Europe;Evolution;Fern;Fish;France;Germany;Glacier;Hip_hop_music;Ichthyosaur;India;Invertebrate;Japan;North_America;Ornithischia;Pterosaur;Reptile;Russia;Saurischia;South_America;Switzerland;Triassic;United_Kingdom;United_States;World_Heritage_Site", "Target": "Cologne"}'}

{'max_tokens

In [461]:
i = 0
for index, row in paths_real_finished_sample.iterrows():
    if i==5:
        break
    print(row['src'], row['tgt'], row['path_len'])
    for conf in ['conf_test1', 'conf_test2']:
        response, tokens_used = navigate_with_memory(row['src'], row['src'], row['tgt'], [row['src']], click_demonstrations, links[row['src']], conf, 0)
        print(payload[conf])
        print("RAW Response", response)
        response_json_obj = extract_json_from_response(response[0])
        print("JSON extracted from response", response_json_obj)
        try:
            gen_click_obj = json.loads(response[0])['Output']
        except ValueError as e:
            print(e, gen_click_obj)
        print(tokens_used)
        print()
    i+=1

Jurassic Cologne 2
In navigate_with_memory: {'role': 'user', 'content': '{"Source": "Jurassic", "Traversed path": "Jurassic", "Current": "Jurassic", "Candidates": "Ammonite;Atlantic_Ocean;Batholith;Canada;Cretaceous;Dinosaur;Europe;Evolution;Fern;Fish;France;Germany;Glacier;Hip_hop_music;Ichthyosaur;India;Invertebrate;Japan;North_America;Ornithischia;Pterosaur;Reptile;Russia;Saurischia;South_America;Switzerland;Triassic;United_Kingdom;United_States;World_Heritage_Site", "Target": "Cologne"}'}

{'max_tokens': 256, 'temperature': 1, 'top_p': 1}
RAW Response ['{"Output": "Germany"}']
JSON extracted from response ['{"Output": "Germany"}']
555

In navigate_with_memory: {'role': 'user', 'content': '{"Source": "Jurassic", "Traversed path": "Jurassic", "Current": "Jurassic", "Candidates": "Ammonite;Atlantic_Ocean;Batholith;Canada;Cretaceous;Dinosaur;Europe;Evolution;Fern;Fish;France;Germany;Glacier;Hip_hop_music;Ichthyosaur;India;Invertebrate;Japan;North_America;Ornithischia;Pterosaur;Reptile;

In [462]:
i = 0
for index, row in paths_real_finished_sample.iterrows():
    if i==5:
        break
    print(row['src'], row['tgt'], row['path_len'])
    for conf in ['conf_test1', 'conf_test2']:
        response, tokens_used = navigate_with_strategies(row['src'], row['src'], row['tgt'], [row['src']], click_demonstrations, links[row['src']], conf, 0)
        print(payload[conf])
        print("RAW Response", response)
        response_json_obj = extract_json_from_response(response[0])
        print("JSON extracted from response", response_json_obj)
        try:
            gen_click_obj = json.loads(response[0])['Output']
        except ValueError as e:
            print(e, gen_click_obj)
        print(tokens_used)
        print()
    i+=1

Jurassic Cologne 2
In navigate_with_memory: {'role': 'user', 'content': '{"Source": "Jurassic", "Traversed path": "Jurassic", "Current": "Jurassic", "Candidates": "Ammonite;Atlantic_Ocean;Batholith;Canada;Cretaceous;Dinosaur;Europe;Evolution;Fern;Fish;France;Germany;Glacier;Hip_hop_music;Ichthyosaur;India;Invertebrate;Japan;North_America;Ornithischia;Pterosaur;Reptile;Russia;Saurischia;South_America;Switzerland;Triassic;United_Kingdom;United_States;World_Heritage_Site", "Target": "Cologne"}'}

{'max_tokens': 256, 'temperature': 1, 'top_p': 1}
RAW Response ['{\n    "Output": "Germany",\n    "thoughts": {\n        "reasoning": "Selecting \'Germany\' as it is the most likely page that could lead us to \'Cologne\'",\n        "plan": [\n            "Explore \'Germany\' page",\n            "Look for links related to \'Cologne\'"\n        ]\n    }\n}']
JSON extracted from response ['{\n    "Output": "Germany",\n    "thoughts": {\n        "reasoning": "Selecting \'Germany\' as it is the most l

In [453]:
# Current system prompt
demonstrations = [{"role":"system", "content": 
             f'''You are a knowledge discovery expert familiar with the Wikipedia link structure and your objective is to play the game of Wikispeedia: https://dlab.epfl.ch/wikispeedia/play/.'''}]

# Old initial prompt for describing the task
# demonstrations.append({"role": "user", "content": '''Given a source article, outgoing links from the source article as candidates, and a target article, your task is to select a link from the provided candidates that takes you closer to the target. 
# If none of the provided candidates takes you closer to the target article, select "NA".'''})

# Current initial prompt for describing the task
demonstrations.append({"role": "user", "content": '''##Goal \nGiven two Wikipedia articles, a source and a target, your goal is to reach the target article starting from the source article. For the articles you are given this is always possible.\n
##Constraint \nYou should exclusively follow the links present in the articles that you encounter along the way.\n
##Fine-grained instructions \n1. While the overall goal is to find a path from a source to a target article, you will proceed step by step.\n2. Given outgoing links from the source article as candidates, you should select the candidate that takes you closer to the target article. Use your knowledge of the "expected" Wikipedia link structure and relatedness between articles to identify the candidate that takes you closer to the target.\n3. If none of the provided candidates takes you closer to the target article, select "NA".\n4. Choose **only** from the provided candidates.\n5. Do not provide an algorithm or code to solve the task, instead present only the solution.\n
##Input \nYou will be given input in the JSON format as described below\n
Input format:
{
    "Source": "<source>", 
    "Candidates": "<candidates>", 
    "Target": "<target>"
}

##Output \nYou should only respond in the JSON format as described below\n
Output format:
{
    "Output": "<output>"
}
Ensure the response can be parsed by Python json.loads
'''})

for entry in demonstrations:
    print(entry['role'])
    print(entry['content'])

system
You are a knowledge discovery expert familiar with the Wikipedia link structure and your objective is to play the game of Wikispeedia: https://dlab.epfl.ch/wikispeedia/play/.
user
##Goal 
Given two Wikipedia articles, a source and a target, your goal is to reach the target article starting from the source article. For the articles you are given this is always possible.

##Constraint 
You should exclusively follow the links present in the articles that you encounter along the way.

##Fine-grained instructions 
1. While the overall goal is to find a path from a source to a target article, you will proceed step by step.
2. Given outgoing links from the source article as candidates, you should select the candidate that takes you closer to the target article. Use your knowledge of the "expected" Wikipedia link structure and relatedness between articles to identify the candidate that takes you closer to the target.
3. If none of the provided candidates takes you closer to the target a

In [454]:
demonstrations = [{"role":"system", "content": 
             f'''You are a knowledge discovery expert familiar with the Wikipedia link structure and your objective is to play the game of Wikispeedia: https://dlab.epfl.ch/wikispeedia/play/.'''}]

# Initial prompt for describing the task
##Performance Evaluation \n1. The shorter the path the better, continuously track the number of articles visited in the "traversed path" part of input.\n2. Analyze the "traversed_path" part of input to reflect on past decisions and strategies to refine your approach.\n
demonstrations.append({"role": "user", "content": '''##Goal \nGiven two Wikipedia articles, a source and a target, your goal is to reach the target article starting from the source article. For the articles you are given this is always possible.\n
##Constraint \nYou should exclusively follow the links present in the articles that you encounter along the way.\n
##Fine-grained instructions \n1. While the overall goal is to find a path from a source to a target article, you will proceed step by step.\n2. Given outgoing links from the current article as candidates, you should select the candidate that takes you closer to the target article. Use your knowledge of the "expected" Wikipedia link structure and relatedness between articles to identify the candidate that takes you closer to the target.\n3. If none of the provided candidates takes you closer to the target article, select "NA".\n4. Choose **only** from the provided candidates.\n5. Do not provide an algorithm or code to solve the task, instead present only the solution.\n
##Performance Evaluation \nAnalyze the "Traversed path" field of input JSON to reflect on past decisions and strategies to refine your approach.\n
##Input \nYou will be given input in the JSON format as described below\n
Input format:
{
    "Source": "<source article>", 
    "Traversed path": "<articles visited so far>", 
    "Current": "<current article>"
    "Candidates": "<outgoing links from current article>", 
    "Target": "<target article>"
}

##Output \nYou should only respond in the JSON format as described below\n
Output format:
{
    "Output": "<output>"
}
Ensure the response can be parsed by Python json.loads
'''})

for entry in demonstrations:
    print(entry['role'])
    print(entry['content'])

system
You are a knowledge discovery expert familiar with the Wikipedia link structure and your objective is to play the game of Wikispeedia: https://dlab.epfl.ch/wikispeedia/play/.
user
##Goal 
Given two Wikipedia articles, a source and a target, your goal is to reach the target article starting from the source article. For the articles you are given this is always possible.

##Constraint 
You should exclusively follow the links present in the articles that you encounter along the way.

##Fine-grained instructions 
1. While the overall goal is to find a path from a source to a target article, you will proceed step by step.
2. Given outgoing links from the current article as candidates, you should select the candidate that takes you closer to the target article. Use your knowledge of the "expected" Wikipedia link structure and relatedness between articles to identify the candidate that takes you closer to the target.
3. If none of the provided candidates takes you closer to the target 

In [455]:
demonstrations = [{"role":"system", "content": 
             f'''You are a knowledge discovery expert familiar with the Wikipedia link structure and your objective is to play the game of Wikispeedia: https://dlab.epfl.ch/wikispeedia/play/.'''}]

# Initial prompt for describing the task
##Performance Evaluation \n1. The shorter the path the better, continuously track the number of articles visited in the "traversed path" part of input.\n2. Analyze the "traversed_path" part of input to reflect on past decisions and strategies to refine your approach.\n
demonstrations.append({"role": "user", "content": '''##Goal \nGiven two Wikipedia articles, a source and a target, your goal is to reach the target article starting from the source article. For the articles you are given this is always possible.\n
##Constraint \nYou should exclusively follow the links present in the articles that you encounter along the way.\n
##Fine-grained instructions \n1. While the overall goal is to find a path from a source to a target article, you will proceed step by step.\n2. Given outgoing links from the current article as candidates, you should select the candidate that takes you closer to the target article. Use your knowledge of the "expected" Wikipedia link structure and relatedness between articles to identify the candidate that takes you closer to the target.\n3. If none of the provided candidates takes you closer to the target article, select "NA".\n4. Choose **only** from the provided candidates.\n5. Do not provide an algorithm or code to solve the task, instead present only the solution.\n
##Performance Evaluation \nAnalyze the "Traversed path" field of input JSON to reflect on past decisions and strategies to refine your approach.\n
##Input \nYou will be given input in the JSON format as described below\n
Input format:
{
    "Source": "<source article>", 
    "Traversed path": "<articles visited so far>", 
    "Current": "<current article>"
    "Candidates": "<outgoing links from current article>", 
    "Target": "<target article>"
}

##Output \nYou should only respond in the JSON format as described below\n
Output format:
{
    "Output": "<output>"
    "thoughts": {
        "reasoning": "<reasoning behind the chosen candidate>",
        "plan": "<- short bulleted\n- list that conveys\n- long-term plan to reach the target>"
    },
}
Ensure the response can be parsed by Python json.loads
'''})

for entry in demonstrations:
    print(entry['role'])
    print(entry['content'])

system
You are a knowledge discovery expert familiar with the Wikipedia link structure and your objective is to play the game of Wikispeedia: https://dlab.epfl.ch/wikispeedia/play/.
user
##Goal 
Given two Wikipedia articles, a source and a target, your goal is to reach the target article starting from the source article. For the articles you are given this is always possible.

##Constraint 
You should exclusively follow the links present in the articles that you encounter along the way.

##Fine-grained instructions 
1. While the overall goal is to find a path from a source to a target article, you will proceed step by step.
2. Given outgoing links from the current article as candidates, you should select the candidate that takes you closer to the target article. Use your knowledge of the "expected" Wikipedia link structure and relatedness between articles to identify the candidate that takes you closer to the target.
3. If none of the provided candidates takes you closer to the target 